In [16]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field

In [17]:
load_dotenv()
model = ChatOpenAI(model='gpt-4o-mini')

In [18]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(description='Detailed feedback for the essay')
    score: int = Field(description = 'Score out of 10', gt=0, le=10)
    
    


In [19]:
structured_output = model.with_structured_output(EvaluationSchema)

In [20]:
essay = """The rapid growth of Artificial Intelligence is changing the future of Computer Science students. Powerful AI models such as GPT Astra and other advanced systems can now generate code, debug programs, analyze data, build applications, and perform many tasks that previously required human programmers.

However, this does not mean that Computer Science will become useless. Instead, the role of CS students will change. In the future, students will need to focus less on memorizing programming syntax and more on problem-solving, system design, software architecture, and understanding how computers actually work.

Basic coding jobs may become more competitive because AI can automate repetitive development tasks. At the same time, new career opportunities are emerging in areas such as AI engineering, machine learning, cybersecurity, cloud computing, data engineering, robotics, and AI agents.

CS students should therefore learn how to use AI as a tool rather than fear it. Strong knowledge of programming, databases, algorithms, operating systems, and networking will still remain important because developers must verify and improve AI-generated solutions.

The future will belong to engineers who can combine technical knowledge, creativity, and AI tools. AI may write more code, but humans will still decide what to build and how to build it correctly.
"""


In [21]:
prompt = f"Evaluate the language quality of the following essay and provide a feedback on scale from 1 to 10"
structured_output.invoke(prompt).score

8

In [22]:
import operator
class CSSState(TypedDict):
    
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    # Merging here inthe individual scores
    individual_scores: Annotated[list[int], operator]
    avg_score: float

In [23]:
def evaluate_language(state:CSSState):
    
    prompt = f"Evaluate the language quality of the essay, provide a feedback and give a score from 0 to 10"
    output = structured_output.invoke(prompt)
    
    return {'language_feedback': output.feedback, 'individual_scores': [output.score]}

In [24]:
def evaluate_analysis(state:CSSState):
    
    prompt = f"Evaluate the depth of analysis of the essay, provide a feedback and give a score from 0 to 10"
    output = structured_output.invoke(prompt)
    
    return {'analysis_feedback': output.feedback, 'individual_scores': [output.score]}

In [25]:
def evaluate_clarity(state:CSSState):
    
    prompt = f"Evaluate the clarity of thought of the essay, provide a feedback and give a score from 0 to 10"
    output = structured_output.invoke(prompt)
    
    return {'clarity_feedback': output.feedback, 'individual_scores': [output.score]}

In [26]:
def final_evaluation(state: CSSState):
    prompt = f"Based on the following feedbacks, create a summarized feedback \n language feedback - {state['language_feedback']} \n depth of analysis feedback - {state['analysis_feedback']} \n clarity of thought - {state['clarity_feedback']}"
    overall_feedback = model.invoke(prompt).content
    
    average = sum(state['individual_scores'])/len(state['individual_scores'])
    
    return {'overall_feedback': overall_feedback, 'avg_score':average}
    

In [ ]:
graph = StateGraph(CSSState)

graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_clarity', evaluate_clarity)
graph.add_node('final_evaluation', final_evaluation)

# Now im adding edges
graph.add_edge(START, "evaluate_analysis")
graph.add_edge(START, "evaluate_language")
graph.add_edge(START, "evaluate_clarity")

graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_clarity', 'final_evaluation')

graph.add_edge('final_evaluation', END)

workflow = graph.compile()
workflow


ValueError: Found edge ending at unknown node `<function evaluate_analysis at 0x000002E894D634C0>`